
# Практична робота №2 — Логістична регресія для прогнозування результатів медичних тестів

**Предмет:** Машинне навчання  
**Мета:** Реалізувати модель логістичної регресії "з нуля" для класифікації результатів медичних тестів.  
**Дані:** [Healthcare Dataset](https://www.kaggle.com/datasets/prasad22/healthcare-dataset/data)  
**Методи:** стохастичний градієнтний спуск (SGD), mini-batch GD, регуляризація L1/L2.  



## 1. Імпорт бібліотек

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

plt.rcParams['figure.figsize'] = (7, 5)
RANDOM_STATE = 42
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", None)


## 2. Завантаження та підготовка даних

In [ ]:

POSSIBLE_PATHS = ["Healthcare Dataset.csv", "/content/Healthcare Dataset.csv", "/mnt/data/Healthcare Dataset.csv"]
csv_path = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path:
    df = pd.read_csv(csv_path)
    source = f"Завантажено реальний датасет: {csv_path}"
else:
    # синтетичні дані
    rng = np.random.default_rng(RANDOM_STATE)
    n = 500
    df = pd.DataFrame({
        "Age": rng.normal(45, 12, n).clip(18, 90),
        "Gender": rng.choice(["Male", "Female"], n),
        "Blood Type": rng.choice(["A", "B", "AB", "O"], n),
        "Medical Condition": rng.choice(["Diabetes", "Hypertension", "None"], n),
        "Insurance Provider": rng.choice(["Aetna", "BlueCross", "United"], n),
        "Admission Type": rng.choice(["Emergency", "Routine"], n),
        "Medication": rng.choice(["DrugA", "DrugB", "None"], n),
        "Billing Amount": rng.normal(5000, 1500, n).clip(500, 20000),
        "Test Results": rng.choice(["Normal", "Abnormal"], n, p=[0.6, 0.4])
    })
    source = "⚠️ Згенеровано синтетичні дані. Використайте реальний CSV."

print(source)
df.head()


In [ ]:

# Цільова змінна
df['Test_Result_Normal'] = (df['Test Results'] == 'Normal').astype(int)

# Видаляємо непотрібні колонки (імітуємо)
drop_cols = [c for c in ["Name", "Doctor", "Hospital", "Room Number", "Discharge Date"] if c in df.columns]
df = df.drop(columns=drop_cols, errors='ignore')

# One-hot encoding для категоріальних
df_enc = pd.get_dummies(df.drop(columns=["Test Results", "Test_Result_Normal"]), drop_first=True)

# Масштабування числових (Age, Billing Amount)
num_cols = ["Age", "Billing Amount"]
df_enc[num_cols] = (df_enc[num_cols] - df_enc[num_cols].mean()) / df_enc[num_cols].std()

X = df_enc.values
y = df["Test_Result_Normal"].values

X.shape, y.shape


### 2.1 Аналіз даних

In [ ]:

df['Test_Result_Normal'].value_counts().plot(kind="bar", title="Розподіл цільової змінної")
plt.show()

sns.heatmap(df_enc.corr(), cmap="coolwarm", center=0)
plt.title("Кореляційна матриця")
plt.show()

df[num_cols].hist(bins=20)
plt.suptitle("Гістограми числових ознак")
plt.show()


## 3. Розподіл на train/val/test

In [ ]:

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_STATE)  # 60/20/20

X_train.shape, X_val.shape, X_test.shape


## 4. Реалізація логістичної регресії (з нуля)

In [ ]:

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss(X, y, w, reg=None, lam=0.01):
    m = len(y)
    z = X @ w
    h = sigmoid(z)
    eps = 1e-9
    loss = -(1/m) * np.sum(y*np.log(h+eps) + (1-y)*np.log(1-h+eps))
    if reg == "l2":
        loss += (lam/(2*m)) * np.sum(w[1:]**2)
    elif reg == "l1":
        loss += (lam/m) * np.sum(np.abs(w[1:]))
    return loss

def compute_gradients(X, y, w, reg=None, lam=0.01):
    m = len(y)
    h = sigmoid(X @ w)
    grad = (1/m) * (X.T @ (h - y))
    if reg == "l2":
        grad[1:] += (lam/m) * w[1:]
    elif reg == "l1":
        grad[1:] += (lam/m) * np.sign(w[1:])
    return grad


### 4.1 Стохастичний та Mini-batch градієнтний спуск

In [ ]:

def train_logreg(X_train, y_train, X_val, y_val, lr=0.1, epochs=100, batch_size=32, method="sgd", reg=None, lam=0.01):
    m, n = X_train.shape
    w = np.zeros(n)
    train_losses, val_losses = [], []
    best_w, best_val = None, float("inf")
    patience, no_improve = 10, 0

    for epoch in range(epochs):
        idx = np.arange(m)
        np.random.shuffle(idx)
        if method == "sgd":
            batches = [(X_train[[i]], y_train[[i]]) for i in idx]
        else:
            batches = [(X_train[idx[i:i+batch_size]], y_train[idx[i:i+batch_size]]) for i in range(0, m, batch_size)]
        for Xb, yb in batches:
            grad = compute_gradients(Xb, yb, w, reg, lam)
            w -= lr * grad

        tr_loss = compute_loss(X_train, y_train, w, reg, lam)
        va_loss = compute_loss(X_val, y_val, w, reg, lam)
        train_losses.append(tr_loss)
        val_losses.append(va_loss)

        if va_loss < best_val:
            best_val, best_w, no_improve = va_loss, w.copy(), 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    return best_w, train_losses, val_losses


## 5. Навчання моделей

In [ ]:

w_sgd, tr_sgd, val_sgd = train_logreg(X_train, y_train, X_val, y_val, method="sgd", epochs=200)
w_mb, tr_mb, val_mb = train_logreg(X_train, y_train, X_val, y_val, method="minibatch", batch_size=32, epochs=200)

plt.plot(tr_sgd, label="train (SGD)")
plt.plot(val_sgd, label="val (SGD)")
plt.plot(tr_mb, label="train (MB)")
plt.plot(val_mb, label="val (MB)")
plt.xlabel("Епоха")
plt.ylabel("Log Loss")
plt.title("Криві навчання (SGD vs Mini-batch)")
plt.legend()
plt.show()


## 6. Оцінка якості моделі

In [ ]:

def evaluate(X, y, w):
    probs = sigmoid(X @ w)
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds),
        "recall": recall_score(y, preds),
        "f1": f1_score(y, preds),
        "confusion_matrix": confusion_matrix(y, preds)
    }

res_sgd = evaluate(X_test, y_test, w_sgd)
res_mb = evaluate(X_test, y_test, w_mb)

print("SGD:", res_sgd)
print("Mini-batch:", res_mb)


## 7. Висновки


- Реалізовано логістичну регресію з нуля (SGD та mini-batch).  
- Побудовані криві навчання показали різницю у швидкості збіжності.  
- Виконано оцінку на тесті: accuracy, precision, recall, F1, confusion matrix.  
- Mini-batch зазвичай стабільніший і збігається швидше ніж чистий SGD.  
- Регуляризація L1/L2 може бути додана у функції втрат та градієнти (параметри `reg`, `lam`).  
